In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DATASET_PATH = os.getenv("DATASET_PATH")

# Dataset preprocessing

In [2]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMOTION_LABEL= ['Anxiety', 'Peace', 'Weariness', 'Happiness', 'Anger']
DRIVER_BEHAVIOR_LABEL = ['Smoking', 'Making Phone', 'Looking Around', 'Dozing Off', 'Normal Driving', 'Talking', 'Body Movement']
SCENE_CENTRIC_CONTEXT_LABEL= ['Traffic Jam', 'Waiting', 'Smooth Traffic']
VEHICLE_BASED_CONTEXT_LABEL= ['Parking', 'Turning', 'Backward Moving', 'Changing Lane', 'Forward Moving']


class CarDataset(Dataset):

    def __init__(self, csv_file, transform=None, **kwargs):
        self.path = pd.read_csv(csv_file)
        # self.path='/root/'+self.path
        self.transform = transform
        self.resize_height = 224
        self.resize_width = 224
        self.body_height = 112
        self.body_width = 112
        self.face_height = 64#56 #64
        self.face_width = 64#56 #64

    # /root/autodl-tmp/AIDE_Dataset/AIDE_Dataset/annotation/0006.json
    def __len__(self):
        return len(self.path)

    # 为数据加载器提供单个样本，包括图像数据和相关标签
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        frames_path, label_path = self.path.iloc[idx]
        frames_path = f"{DATASET_PATH}/{frames_path}"
        label_path = f"{DATASET_PATH}/{label_path}"

        parts1 = frames_path.split('/')
        # parts1.insert(4, 'AIDE_Dataset')  # 在第四个元素后添加 'AIDE_Dataset'
        frames_path = '/'.join(parts1)

        parts2 = label_path.split('/')
        # parts2.insert(4, 'AIDE_Dataset')  # 在第四个元素后添加 'AIDE_Dataset'
        label_path = '/'.join(parts2)

        label_json = json.load(open(label_path))
        pose_list = label_json['pose_list']

        # buffer, buffer_front, buffer_left, buffer_right, buffer_face, buffer_body,keypoints = self.load_frames(frames_path, pose_list)  # 加载图像帧数据
        buffer, buffer_front, buffer_left, buffer_right, buffer_face, buffer_body, posture, gesture = self.load_frames(frames_path, pose_list)

        # buffer, buffer_front, buffer_left, buffer_right, buffer_body, buffer_face, keypoints = self.load_frames(
        #     frames_path, pose_list)  # 加载图像帧数据

        # 数据增强
        buffer = self.randomflip(buffer)
        buffer_front = self.randomflip(buffer_front)
        buffer_left = self.randomflip(buffer_left)
        buffer_right = self.randomflip(buffer_right)

        context = torch.cat([buffer, buffer_front, buffer_left, buffer_right], dim=0)  # 将四个张量沿批次维度拼接
        context = self.to_tensor(context)

        # 加载的图像数据-->PyTorch张量
        buffer = self.to_tensor(buffer)
        buffer_front = self.to_tensor(buffer_front)
        buffer_left = self.to_tensor(buffer_left)
        buffer_right = self.to_tensor(buffer_right)

        # 身体、面部、关节点
        buffer_body = self.to_tensor(buffer_body)
        buffer_face = self.to_tensor(buffer_face)
        # keypoints = keypoints.permute(2, 0, 1).contiguous()

        emotion_label = EMOTION_LABEL.index((label_json['emotion_label'].capitalize()))
        driver_behavior_label = DRIVER_BEHAVIOR_LABEL.index((label_json['driver_behavior_label']))
        scene_centric_context_label = SCENE_CENTRIC_CONTEXT_LABEL.index((label_json['scene_centric_context_label']))

        # 标签错误情况
        if label_json['vehicle_based_context_label'] == "Forward":
            label_json['vehicle_based_context_label'] = "Forward Moving"
        # print(label_json['vehicle_based_context_label'], label_path)
        vehicle_based_context_label = VEHICLE_BASED_CONTEXT_LABEL.index((label_json['vehicle_based_context_label']))

        sample = {
            'context': context,
            'body': buffer_body,
            'face': buffer_face,
            # 'keypoints': torch.stack([keypoints], dim=-1),
            'posture': posture,  # 使用 posture
            'gesture': gesture,  # 使用 gesture
            "emotion_label": emotion_label,
            "driver_behavior_label": driver_behavior_label,
            "scene_centric_context_label": scene_centric_context_label,
            "vehicle_based_context_label": vehicle_based_context_label
        }

        # keypoints = sample['keypoints']
        posture = sample['posture']  # 使用 posture
        gesture = sample['gesture']  # 使用 gesture
        context = sample['context']
        body = sample['body']
        face = sample['face']
        emotion_label = sample['emotion_label']
        behavior_label = sample['driver_behavior_label']
        context_label = sample['scene_centric_context_label']
        vehicle_label = sample['vehicle_based_context_label']
      

        # 返回图像数据和相关标签
        return buffer, buffer_front, buffer_left, buffer_right, buffer_face, buffer_body, posture, gesture, emotion_label, behavior_label, context_label, vehicle_label


    def load_frames(self, file_dir, pose_list):

        incar_path = os.path.join(file_dir, 'incarframes')
        front_frames = os.path.join(file_dir, 'frontframes')
        left_frames = os.path.join(file_dir, 'leftframes')
        right_frames = os.path.join(file_dir, 'rightframes')
        face_frames = os.path.join(file_dir, 'face')
        body_frames = os.path.join(file_dir, 'body')


        frames = [os.path.join(incar_path, img) for img in os.listdir(incar_path) if img.endswith('.jpg')]
        front_frames = [os.path.join(front_frames, img) for img in os.listdir(front_frames) if img.endswith('.jpg')]
        left_frames = [os.path.join(left_frames, img) for img in os.listdir(left_frames) if img.endswith('.jpg')]
        right_frames = [os.path.join(right_frames, img) for img in os.listdir(right_frames) if img.endswith('.jpg')]

        face_frames = [os.path.join(face_frames, img) for img in os.listdir(face_frames) if img.endswith('.jpg')]
        if len(face_frames)!=45:
            face_frames.extend([face_frames[-1]] * (45 - len(face_frames)))
        body_frames = [os.path.join(body_frames, img) for img in os.listdir(body_frames) if img.endswith('.jpg')]
        if len(body_frames)!=45:
            body_frames.extend([body_frames[-1]] * (45 - len(body_frames)))


        frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))
        front_frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))
        left_frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))
        right_frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))
        face_frames.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))
        body_frames.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))




        buffer, buffer_front, buffer_left, buffer_right, keypoints_list, buffer_face, buffer_body= [], [], [], [], [], [], []
        posture_list, gesture_list = [], [] 

        for i, frame_name in enumerate(frames):
            if not i == 0 and not i % 3 == 2:
                continue
            if i >= 45:
                break

            img = cv2.imread(frame_name)
            front_img = cv2.imread(front_frames[i])
            left_img = cv2.imread(left_frames[i])
            right_img = cv2.imread(right_frames[i])

            img_face = cv2.imread(face_frames[i])
            img_body = cv2.imread(body_frames[i])
            keypoints = np.array(pose_list[i]['result'][0]['keypoints']).reshape(-1, 3)
            # keypoints_list.append(torch.from_numpy(keypoints).float())
            # keypoint=keypoint[94:115]
            # posture =  keypoint[:,:,:,:26,:]
            # gesture = keypoint[:,:,:,94:,:]            
            posture =  keypoints[:26]
            gesture = keypoints[94:136]
            
            posture_list.append(posture)
            gesture_list.append(gesture)
            
            
            
            # img_body = img[int(body[1]):int(body[1] + max(body[3], 20)), int(body[0]):int(body[0] + max(body[2], 10))]
            # img_face = img[int(face[1]):int(face[1] + max(face[3], 10)), int(face[0]):int(face[0] + max(face[2], 10))]

            if img.shape[0] != self.resize_height or img.shape[1] != self.resize_width:
                img = cv2.resize(img, (self.resize_width, self.resize_height))
            if front_img.shape[0] != self.resize_height or front_img.shape[1] != self.resize_width:
                front_img = cv2.resize(front_img, (self.resize_width, self.resize_height))
            if left_img.shape[0] != self.resize_height or left_img.shape[1] != self.resize_width:
                left_img = cv2.resize(left_img, (self.resize_width, self.resize_height))
            if right_img.shape[0] != self.resize_height or right_img.shape[1] != self.resize_width:
                right_img = cv2.resize(right_img, (self.resize_width, self.resize_height))

            if img_body.shape[0] != self.resize_height or img_body.shape[1] != self.resize_width:
                img_body = cv2.resize(img_body, (self.resize_width, self.resize_height))

            try:
                if img_face.shape[0] != self.face_height or img_face.shape[1] != self.face_width:
                    img_face = cv2.resize(img_face, (self.face_width, self.face_height))
                # if img_face.shape[0] != self.resize_height or img_face.shape[1] != self.resize_width:
                #     img_face = cv2.resize(img_face, (self.resize_width, self.resize_height))
            except:
                img_face = img_body

            buffer.append(torch.from_numpy(img).float())
            buffer_front.append(torch.from_numpy(front_img).float())
            buffer_left.append(torch.from_numpy(left_img).float())
            buffer_right.append(torch.from_numpy(right_img).float())

            buffer_body.append(torch.from_numpy(img_body).float())
            # if len(face_frames)==45:
            buffer_face.append(torch.from_numpy(img_face).float())
            # keypoints.append(torch.from_numpy(keypoints).float())
            # keypoints_tensor = torch.stack(keypoints_list)
            # posture_tensor = torch.tensor(posture_list, dtype=torch.float)
            # gesture_tensor = torch.tensor(gesture_list, dtype=torch.float)

            posture_array = np.array(posture_list, dtype=np.float32)  # 将 posture_list 轈换为 numpy 数组
            gesture_array = np.array(gesture_list, dtype=np.float32)  # 将 gesture_list 转换为 numpy 数组
            posture_tensor = torch.from_numpy(posture_array)  # 将 numpy 数组转换为 PyTorch 张量
            gesture_tensor = torch.from_numpy(gesture_array)  # 将 numpy 数组转换为 PyTorch 张量

        return torch.stack(buffer), torch.stack(buffer_front), torch.stack(buffer_left), torch.stack(
            buffer_right), torch.stack(buffer_face), torch.stack(buffer_body), posture_tensor, gesture_tensor
    #keypoints_tensor#torch.stack(keypoints) 

    # 随机翻转输入的PyTorch张量buffer(数据增强操作)
    def randomflip(self, buffer):

        # 以50%的概率在第二个维度上进行水平翻转
        if np.random.random() < 0.5:
            buffer = torch.flip(buffer, dims=[1])

        # 以50%的概率在第三个维度上进行垂直翻转
        if np.random.random() < 0.5:
            buffer = torch.flip(buffer, dims=[2])

        # 返回翻转后的张量buffer
        return buffer

    def normalize(self, buffer):
        for i, frame in enumerate(buffer):
            frame -= np.array([[[90.0, 98.0, 102.0]]])
            buffer[i] = frame

        return buffer

    def to_tensor(self, buffer):

        return buffer.permute(3, 0, 1, 2).contiguous()

In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader


# Important when using several DataLoader workers:
# prevent every OpenCV worker from spawning its own thread pool.
cv2.setNumThreads(0)


EMOTION_LABEL = [
    "Anxiety",
    "Peace",
    "Weariness",
    "Happiness",
    "Anger",
]

DRIVER_BEHAVIOR_LABEL = [
    "Smoking",
    "Making Phone",
    "Looking Around",
    "Dozing Off",
    "Normal Driving",
    "Talking",
    "Body Movement",
]

SCENE_CENTRIC_CONTEXT_LABEL = [
    "Traffic Jam",
    "Waiting",
    "Smooth Traffic",
]

VEHICLE_BASED_CONTEXT_LABEL = [
    "Parking",
    "Turning",
    "Backward Moving",
    "Changing Lane",
    "Forward Moving",
]


class CarDataset(Dataset):

    def __init__(
        self,
        csv_file,
        dataset_path,
        horizontal_flip_prob=0.0,
        vertical_flip_prob=0.0,
    ):
        """
        Parameters
        ----------
        csv_file:
            CSV containing:
                sample_directory, annotation_json

        dataset_path:
            Root directory containing AIDE_Dataset.

        horizontal_flip_prob:
            Probability of applying one coherent horizontal flip
            to the complete sample.

            Keep at 0.0 while reproducing/debugging the baseline.
            You can later set it to 0.5 if desired.
        """

        # If the official CSV has no header, use header=None.
        self.path = pd.read_csv(csv_file, header=None)

        self.dataset_path = dataset_path
        self.horizontal_flip_prob = horizontal_flip_prob
        self.vertical_flip_prob = vertical_flip_prob

        self.resize_height = 224
        self.resize_width = 224

        self.body_height = 224
        self.body_width = 224

        self.face_height = 64
        self.face_width = 64

        # The original code effectively selects:
        # 0, 2, 5, 8, ..., 44
        #
        # -> 16 frames
        self.selected_indices = [0] + list(range(2, 45, 3))

        assert len(self.selected_indices) == 16

        # Cache directory listings so that they are not repeatedly
        # os.listdir()'d and sorted every epoch.
        #
        # Each persistent DataLoader worker will maintain its own cache.
        self._path_cache = {}


    def __len__(self):
        return len(self.path)


    def __getitem__(self, idx):

        if torch.is_tensor(idx):
            idx = idx.item()

        frames_rel_path = self.path.iloc[idx, 0]
        label_rel_path = self.path.iloc[idx, 1]

        frames_path = os.path.join(
            self.dataset_path,
            str(frames_rel_path),
        )

        label_path = os.path.join(
            self.dataset_path,
            str(label_rel_path),
        )

        # ------------------------------------------------------------
        # Load annotation
        # ------------------------------------------------------------

        with open(label_path, "r") as f:
            label_json = json.load(f)

        pose_list = label_json["pose_list"]

        (
            buffer_incar,
            buffer_front,
            buffer_left,
            buffer_right,
            buffer_face,
            buffer_body,
            posture,
            gesture,
        ) = self.load_frames(
            frames_path,
            pose_list,
        )


        if (
            self.horizontal_flip_prob > 0
            and np.random.random() < self.horizontal_flip_prob
        ):
            (
                buffer_incar,
                buffer_front,
                buffer_left,
                buffer_right,
                buffer_face,
                buffer_body,
            ) = self.horizontal_flip(
                buffer_incar,
                buffer_front,
                buffer_left,
                buffer_right,
                buffer_face,
                buffer_body,
            )

        if (
            self.vertical_flip_prob > 0
            and np.random.random() < self.vertical_flip_prob
        ):
            (
                buffer_incar,
                buffer_front,
                buffer_left,
                buffer_right,
                buffer_face,
                buffer_body,
            ) = self.vertical_flip(
                buffer_incar,
                buffer_front,
                buffer_left,
                buffer_right,
                buffer_face,
                buffer_body,
            )

        buffer_incar = self.to_tensor(buffer_incar)
        buffer_front = self.to_tensor(buffer_front)
        buffer_left = self.to_tensor(buffer_left)
        buffer_right = self.to_tensor(buffer_right)

        buffer_face = self.to_tensor(buffer_face)
        buffer_body = self.to_tensor(buffer_body)

        emotion_label = EMOTION_LABEL.index(
            label_json["emotion_label"].capitalize()
        )

        behavior_label = DRIVER_BEHAVIOR_LABEL.index(
            label_json["driver_behavior_label"]
        )

        context_label = SCENE_CENTRIC_CONTEXT_LABEL.index(
            label_json["scene_centric_context_label"]
        )

        vehicle_name = label_json["vehicle_based_context_label"]

        if vehicle_name == "Forward":
            vehicle_name = "Forward Moving"

        vehicle_label = VEHICLE_BASED_CONTEXT_LABEL.index(
            vehicle_name
        )
        return (
            buffer_incar,
            buffer_front,
            buffer_left,
            buffer_right,
            buffer_face,
            buffer_body,
            posture,
            gesture,
            emotion_label,
            behavior_label,
            context_label,
            vehicle_label,
        )

    def load_frames(self, file_dir, pose_list):

        paths = self.get_frame_paths(file_dir)

        buffer_incar = []
        buffer_front = []
        buffer_left = []
        buffer_right = []

        buffer_face = []
        buffer_body = []

        posture_list = []
        gesture_list = []

        for i in self.selected_indices:

            img_incar = self.read_image(paths["incar"][i])
            img_front = self.read_image(paths["front"][i])
            img_left = self.read_image(paths["left"][i])
            img_right = self.read_image(paths["right"][i])

            img_face = self.read_image(
                paths["face"][i],
                allow_failure=True,
            )

            img_body = self.read_image(
                paths["body"][i],
                allow_failure=True,
            )

            if img_body is None:
                raise RuntimeError(
                    f"Could not load body image for sample "
                    f"{file_dir}, frame {i}"
                )

            if img_face is None:
                img_face = img_body.copy()

            keypoints = np.asarray(
                pose_list[i]["result"][0]["keypoints"],
                dtype=np.float32,
            ).reshape(-1, 3)

            posture_list.append(keypoints[:26])
            gesture_list.append(keypoints[94:136])


            img_incar = self.resize_if_needed(
                img_incar,
                self.resize_width,
                self.resize_height,
            )

            img_front = self.resize_if_needed(
                img_front,
                self.resize_width,
                self.resize_height,
            )

            img_left = self.resize_if_needed(
                img_left,
                self.resize_width,
                self.resize_height,
            )

            img_right = self.resize_if_needed(
                img_right,
                self.resize_width,
                self.resize_height,
            )

            img_body = self.resize_if_needed(
                img_body,
                self.body_width,
                self.body_height,
            )

            img_face = self.resize_if_needed(
                img_face,
                self.face_width,
                self.face_height,
            )


            buffer_incar.append(img_incar)
            buffer_front.append(img_front)
            buffer_left.append(img_left)
            buffer_right.append(img_right)

            buffer_face.append(img_face)
            buffer_body.append(img_body)


        buffer_incar = np.stack(buffer_incar, axis=0)
        buffer_front = np.stack(buffer_front, axis=0)
        buffer_left = np.stack(buffer_left, axis=0)
        buffer_right = np.stack(buffer_right, axis=0)

        buffer_face = np.stack(buffer_face, axis=0)
        buffer_body = np.stack(buffer_body, axis=0)

        posture = torch.from_numpy(
            np.stack(posture_list, axis=0)
        )

        gesture = torch.from_numpy(
            np.stack(gesture_list, axis=0)
        )

        return (
            buffer_incar,
            buffer_front,
            buffer_left,
            buffer_right,
            buffer_face,
            buffer_body,
            posture,
            gesture,
        )



    def get_frame_paths(self, file_dir):

        if file_dir in self._path_cache:
            return self._path_cache[file_dir]

        paths = {
            "incar": self.sorted_jpgs(
                os.path.join(file_dir, "incarframes")
            ),
            "front": self.sorted_jpgs(
                os.path.join(file_dir, "frontframes")
            ),
            "left": self.sorted_jpgs(
                os.path.join(file_dir, "leftframes")
            ),
            "right": self.sorted_jpgs(
                os.path.join(file_dir, "rightframes")
            ),
            "face": self.sorted_jpgs(
                os.path.join(file_dir, "face"),
                underscore_names=True,
            ),
            "body": self.sorted_jpgs(
                os.path.join(file_dir, "body"),
                underscore_names=True,
            ),
        }

        paths["face"] = self.pad_paths(paths["face"], 45)
        paths["body"] = self.pad_paths(paths["body"], 45)

        for name in ("incar", "front", "left", "right"):
            if len(paths[name]) < 45:
                raise RuntimeError(
                    f"{file_dir}: expected at least 45 "
                    f"{name} frames, got {len(paths[name])}"
                )

        self._path_cache[file_dir] = paths

        return paths


    @staticmethod
    def sorted_jpgs(folder, underscore_names=False):

        files = [
            os.path.join(folder, filename)
            for filename in os.listdir(folder)
            if filename.lower().endswith(".jpg")
        ]

        if underscore_names:
            files.sort(
                key=lambda p: int(
                    os.path.basename(p).split("_")[0]
                )
            )
        else:
            files.sort(
                key=lambda p: int(
                    os.path.splitext(os.path.basename(p))[0]
                )
            )

        return files


    @staticmethod
    def pad_paths(paths, target_length):

        if len(paths) == 0:
            raise RuntimeError(
                "No images found in a required face/body directory."
            )

        if len(paths) < target_length:
            paths = paths + [paths[-1]] * (
                target_length - len(paths)
            )

        return paths

    @staticmethod
    def read_image(path, allow_failure=False):

        img = cv2.imread(
            path,
            cv2.IMREAD_COLOR,
        )

        if img is None and not allow_failure:
            raise RuntimeError(
                f"Could not read image: {path}"
            )

        return img


    @staticmethod
    def resize_if_needed(img, width, height):

        if (
            img.shape[1] != width
            or img.shape[0] != height
        ):
            img = cv2.resize(
                img,
                (width, height),
                interpolation=cv2.INTER_LINEAR,
            )

        return img



    @staticmethod
    def horizontal_flip(
        incar,
        front,
        left,
        right,
        face,
        body,
    ):
        """
        Apply ONE decision to the whole visual sample.

        Axis 2 is width because arrays are THWC.
        """

        return tuple(
            np.ascontiguousarray(
                np.flip(x, axis=2)
            )
            for x in (
                incar,
                front,
                left,
                right,
                face,
                body,
            )
        )

    @staticmethod
    def vertical_flip(
        incar,
        front,
        left,
        right,
        face,
        body,
    ):
        """
        Apply ONE decision to the whole visual sample.

        Axis 2 is width because arrays are THWC.
        """

        return tuple(
            np.ascontiguousarray(
                np.flip(x, axis=1)
            )
            for x in (
                incar,
                front,
                left,
                right,
                face,
                body,
            )
        )


    @staticmethod
    def to_tensor(buffer):
        """
        THWC -> CTHW.

        Remains uint8 here.
        """
        return (
            torch.from_numpy(buffer)
            .permute(3, 0, 1, 2)
            .contiguous()
        )

In [3]:
train_dataset = CarDataset(csv_file=f'{DATASET_PATH}/training.csv', dataset_path=DATASET_PATH, horizontal_flip_prob=0.5, vertical_flip_prob=0.5)
val_dataset = CarDataset(csv_file=f'{DATASET_PATH}/validation.csv', dataset_path=DATASET_PATH, horizontal_flip_prob=0.0, vertical_flip_prob=0.0)
test_dataset = CarDataset(csv_file=f'{DATASET_PATH}/testing.csv', dataset_path=DATASET_PATH, horizontal_flip_prob=0.0, vertical_flip_prob=0.0)

train_dataloader = DataLoader(train_dataset, batch_size=12, shuffle=True, drop_last=False, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=6, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=6, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)

run_dataset_test = False
if run_dataset_test:
    for i_batch, sample in enumerate(train_dataloader):
        buffer, buffer_front, buffer_left, buffer_right, buffer_face, buffer_body, posture, gesture, emotion_label, behavior_label, context_label, vehicle_label = sample 
        print('buffer:{}, buffer_front:{}, buffer_left:{}, buffer_right:{}, buffer_face:{}, buffer_body:{}, posture:{}, gesture:{}, emotion_label:{}, behavior_label:{}, context_label:{}, vehicle_label:{}' \
    .format(buffer.shape, buffer_front.shape, buffer_left.shape, buffer_right.shape, buffer_face.shape, buffer_body.shape, posture.shape, gesture.shape, emotion_label.shape, behavior_label.shape, context_label.shape, vehicle_label.shape))

# Model definition

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import math

## Multi axis Attention

In [5]:
class qkv_transform(nn.Conv1d):
    """Conv1d for qkv_transform"""

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class AxialAttention(nn.Module):
    def __init__(self, in_planes, out_planes, groups=8, kernel_size=56,
                 stride=1, bias=False, width=False):
        assert (in_planes % groups == 0) and (out_planes % groups == 0)
        super(AxialAttention, self).__init__()
        self.in_planes = in_planes
        self.out_planes = out_planes
        self.groups = groups
        self.group_planes = out_planes // groups
        self.kernel_size = kernel_size
        self.stride = stride
        self.bias = bias
        self.width = width

        # Multi-head self attention
        self.qkv_transform = qkv_transform(in_planes, out_planes * 2, kernel_size=1, stride=1,
                                           padding=0, bias=False)
        self.bn_qkv = nn.BatchNorm1d(out_planes * 2)
        self.bn_similarity = nn.BatchNorm2d(groups * 3)
        # self.bn_qk = nn.BatchNorm2d(groups)
        # self.bn_qr = nn.BatchNorm2d(groups)
        # self.bn_kr = nn.BatchNorm2d(groups)
        self.bn_output = nn.BatchNorm1d(out_planes * 2)

        # Position embedding
        self.relative = nn.Parameter(torch.randn(self.group_planes * 2, kernel_size * 2 - 1), requires_grad=True)
        query_index = torch.arange(kernel_size).unsqueeze(0)
        key_index = torch.arange(kernel_size).unsqueeze(1)
        relative_index = key_index - query_index + kernel_size - 1
        self.register_buffer('flatten_index', relative_index.view(-1))
        if stride > 1:
            self.pooling = nn.AvgPool2d(stride, stride=stride)

        self.reset_parameters()

    def forward(self, x):
            if self.width:
                x = x.permute(0, 2, 1, 3)
            else:
                x = x.permute(0, 3, 1, 2)  # N, W, C, H
            N, W, C, H = x.shape
            x = x.contiguous().view(N * W, C, H)
    
            # Transformations
            qkv = self.bn_qkv(self.qkv_transform(x))
            q, k, v = torch.split(qkv.reshape(N * W, self.groups, self.group_planes * 2, H),
                                  [self.group_planes // 2, self.group_planes // 2, self.group_planes], dim=2)
    
            # Calculate position embedding
            all_embeddings = torch.index_select(self.relative, 1, self.flatten_index).view(self.group_planes * 2,
                                                                                           self.kernel_size,
                                                                                           self.kernel_size)
            q_embedding, k_embedding, v_embedding = torch.split(all_embeddings,
                                                                [self.group_planes // 2, self.group_planes // 2,
                                                                 self.group_planes], dim=0)
            qr = torch.einsum('bgci,cij->bgij', q, q_embedding)
            kr = torch.einsum('bgci,cij->bgij', k, k_embedding).transpose(2, 3)
            qk = torch.einsum('bgci, bgcj->bgij', q, k)
            stacked_similarity = torch.cat([qk, qr, kr], dim=1)
            stacked_similarity = self.bn_similarity(stacked_similarity).view(N * W, 3, self.groups, H, H).sum(dim=1)
            # stacked_similarity = self.bn_qr(qr) + self.bn_kr(kr) + self.bn_qk(qk)
            # (N, groups, H, H, W)
            similarity = F.softmax(stacked_similarity, dim=3)
            sv = torch.einsum('bgij,bgcj->bgci', similarity, v)
            sve = torch.einsum('bgij,cij->bgci', similarity, v_embedding)
            stacked_output = torch.cat([sv, sve], dim=-1).view(N * W, self.out_planes * 2, H)
            output = self.bn_output(stacked_output).view(N, W, self.out_planes, 2, H).sum(dim=-2)
    
            if self.width:
                output = output.permute(0, 2, 1, 3)
            else:
                output = output.permute(0, 2, 3, 1)
    
            if self.stride > 1:
                output = self.pooling(output)
    
            return output

    def reset_parameters(self):
        self.qkv_transform.weight.data.normal_(0, math.sqrt(1. / self.in_planes))
        # nn.init.uniform_(self.relative, -0.1, 0.1)
        nn.init.normal_(self.relative, 0., math.sqrt(1. / self.group_planes))


class AxialBlock(nn.Module):
    expansion = 2

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None, kernel_size=56):
        super(AxialBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.))
        # Both self.conv2 and self.downsample layers downsample the input when stride != 1
        self.conv_down = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.hight_block = AxialAttention(width, width, groups=groups, kernel_size=kernel_size)
        self.width_block = AxialAttention(width, width, groups=groups, kernel_size=kernel_size, stride=stride,
                                          width=True)
        self.conv_up = conv1x1(width, planes * self.expansion)
        self.bn2 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv_down(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.hight_block(out)
        out = self.width_block(out)
        out = self.relu(out)

        out = self.conv_up(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class AxialAttentionNet(nn.Module):

    def __init__(self, block, layers, num_classes=1000, zero_init_residual=True,
                 groups=8, width_per_group=64, replace_stride_with_dilation=None,
                 norm_layer=None, s=1):
        super(AxialAttentionNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = int(64 * s)
        self.dilation = 1
        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError("replace_stride_with_dilation should be None "
                             "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
        self.groups = groups
        self.base_width = width_per_group
        # 改：   # 添加一个新的卷积层来降低通道数
        # self.conv_reduce = nn.Conv2d(48, 3, kernel_size=1, stride=1, padding=0, bias=False)
        # self.bn_reduce = nn.BatchNorm2d(3)
        # self.conv1 = nn.Conv2d(48, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)

        self.conv1 = nn.Conv2d(48, self.inplanes, kernel_size=7, stride=2, padding=3,
                               bias=False)

        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, int(128 * s), layers[0], kernel_size=56)
        self.layer2 = self._make_layer(block, int(256 * s), layers[1], stride=2, kernel_size=56,
                                       dilate=replace_stride_with_dilation[0])

        self.layer3 = self._make_layer(block, int(512 * s), layers[2], stride=2, kernel_size=28,
                                       dilate=replace_stride_with_dilation[1])

        self.avgpool = nn.AdaptiveAvgPool2d( (14, 14) )
        self.fc = nn.Linear(14,14)

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Conv1d)):
                if isinstance(m, qkv_transform):
                    pass
                else:
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, AxialBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, kernel_size=56, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, groups=self.groups,
                            base_width=self.base_width, dilation=previous_dilation,
                            norm_layer=norm_layer, kernel_size=kernel_size))
        self.inplanes = planes * block.expansion
        if stride != 1:
            kernel_size = kernel_size // 2

        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer, kernel_size=kernel_size))

        return nn.Sequential(*layers)

    def _forward_impl(self, x):
        # See note [TorchScript super()]

        # x = self.conv_reduce(x)
        # x = self.bn_reduce(x)
        # x = self.relu(x)

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = self.fc(x)

        return x

    def forward(self, x):
        return self._forward_impl(x)



def HVAttention(pretrained=False, **kwargs):
    model = AxialAttentionNet(AxialBlock, [1, 2, 4, 1], s=0.5, **kwargs)
    return model

## Regional attention

In [6]:
from typing import Tuple

from einops import rearrange

class TopkRouting(nn.Module):
    """
    differentiable topk routing with scaling
    Args:
        qk_dim: int, feature dimension of query and key
        topk: int, the 'topk'
        qk_scale: int or None, temperature (multiply) of softmax activation
        with_param: bool, wether inorporate learnable params in routing unit
        diff_routing: bool, wether make routing differentiable
        soft_routing: bool, wether make output value multiplied by routing weights
    """
    def __init__(self, qk_dim, topk=4, qk_scale=None, param_routing=False, diff_routing=False):
        super().__init__()
        self.topk = topk
        self.qk_dim = qk_dim
        self.scale = qk_scale or qk_dim ** -0.5
        self.diff_routing = diff_routing
        # TODO: norm layer before/after linear?
        self.emb = nn.Linear(qk_dim, qk_dim) if param_routing else nn.Identity()
        # routing activation
        self.routing_act = nn.Softmax(dim=-1)
    
    def forward(self, query:Tensor, key:Tensor)->Tuple[Tensor]:
        """
        Args:
            q, k: (n, p^2, c) tensor
        Return:
            r_weight, topk_index: (n, p^2, topk) tensor
        """
        if not self.diff_routing:
            query, key = query.detach(), key.detach()
        query_hat, key_hat = self.emb(query), self.emb(key) # per-window pooling -> (n, p^2, c) 
        attn_logit = (query_hat*self.scale) @ key_hat.transpose(-2, -1) # (n, p^2, p^2)
        topk_attn_logit, topk_index = torch.topk(attn_logit, k=self.topk, dim=-1) # (n, p^2, k), (n, p^2, k)
        r_weight = self.routing_act(topk_attn_logit) # (n, p^2, k)
        
        return r_weight, topk_index

class QKVLinear(nn.Module):
    def __init__(self, dim, qk_dim, bias=True):
        super().__init__()
        self.dim = dim
        self.qk_dim = qk_dim
        self.qkv = nn.Linear(dim, qk_dim + qk_dim + dim, bias=bias)
    
    def forward(self, x):
        q, kv = self.qkv(x).split([self.qk_dim, self.qk_dim+self.dim], dim=-1)
        return q, kv
        # q, k, v = self.qkv(x).split([self.qk_dim, self.qk_dim, self.dim], dim=-1)
        # return q, k, v

class KVGather(nn.Module):
    def __init__(self, mul_weight='none'):
        super().__init__()
        assert mul_weight in ['none', 'soft', 'hard']
        self.mul_weight = mul_weight

    def forward(self, r_idx:Tensor, r_weight:Tensor, kv:Tensor):
        """
        r_idx: (n, p^2, topk) tensor
        r_weight: (n, p^2, topk) tensor
        kv: (n, p^2, w^2, c_kq+c_v)

        Return:
            (n, p^2, topk, w^2, c_kq+c_v) tensor
        """
        # select kv according to routing index
        n, p2, w2, c_kv = kv.size()
        topk = r_idx.size(-1)
        # print(r_idx.size(), r_weight.size())
        # FIXME: gather consumes much memory (topk times redundancy), write cuda kernel? 
        topk_kv = torch.gather(kv.view(n, 1, p2, w2, c_kv).expand(-1, p2, -1, -1, -1), # (n, p^2, p^2, w^2, c_kv) without mem cpy
                                dim=2,
                                index=r_idx.view(n, p2, topk, 1, 1).expand(-1, -1, -1, w2, c_kv) # (n, p^2, k, w^2, c_kv)
                               )

        if self.mul_weight == 'soft':
            topk_kv = r_weight.view(n, p2, topk, 1, 1) * topk_kv # (n, p^2, k, w^2, c_kv)
        elif self.mul_weight == 'hard':
            raise NotImplementedError('differentiable hard routing TBA')
        # else: #'none'
        #     topk_kv = topk_kv # do nothing

        return topk_kv

class BiLevelRoutingAttention(nn.Module):
    """
    n_win: number of windows in one side (so the actual number of windows is n_win*n_win)
    kv_per_win: for kv_downsample_mode='ada_xxxpool' only, number of key/values per window. Similar to n_win, the actual number is kv_per_win*kv_per_win.
    topk: topk for window filtering
    param_attention: 'qkvo'-linear for q,k,v and o, 'none': param free attention
    param_routing: extra linear for routing
    diff_routing: wether to set routing differentiable
    soft_routing: wether to multiply soft routing weights 
    """
    def __init__(self, dim, n_win=7, num_heads=8, qk_dim=None, qk_scale=None,
                 kv_per_win=4, kv_downsample_ratio=4, kv_downsample_kernel=None, kv_downsample_mode='identity',
                 topk=4, param_attention="qkvo", param_routing=False, diff_routing=False, soft_routing=False, side_dwconv=3,
                 auto_pad=True):
        super().__init__()
        # local attention setting
        self.dim = dim
        self.n_win = n_win  # Wh, Ww
        self.num_heads = num_heads
        self.qk_dim = qk_dim or dim
        assert self.qk_dim % num_heads == 0 and self.dim % num_heads==0, 'qk_dim and dim must be divisible by num_heads!'
        self.scale = qk_scale or self.qk_dim ** -0.5


        ################side_dwconv (i.e. LCE in ShuntedTransformer)###########
        self.lepe = nn.Conv2d(dim, dim, kernel_size=side_dwconv, stride=1, padding=side_dwconv//2, groups=dim) if side_dwconv > 0 else \
                    lambda x: torch.zeros_like(x)
        
        ################ global routing setting #################
        self.topk = topk
        self.param_routing = param_routing
        self.diff_routing = diff_routing
        self.soft_routing = soft_routing
        # router
        assert not (self.param_routing and not self.diff_routing) # cannot be with_param=True and diff_routing=False
        self.router = TopkRouting(qk_dim=self.qk_dim,
                                  qk_scale=self.scale,
                                  topk=self.topk,
                                  diff_routing=self.diff_routing,
                                  param_routing=self.param_routing)
        if self.soft_routing: # soft routing, always diffrentiable (if no detach)
            mul_weight = 'soft'
        elif self.diff_routing: # hard differentiable routing
            mul_weight = 'hard'
        else:  # hard non-differentiable routing
            mul_weight = 'none'
        self.kv_gather = KVGather(mul_weight=mul_weight)

        # qkv mapping (shared by both global routing and local attention)
        self.param_attention = param_attention
        if self.param_attention == 'qkvo':
            self.qkv = QKVLinear(self.dim, self.qk_dim)
            self.wo = nn.Linear(dim, dim)
        elif self.param_attention == 'qkv':
            self.qkv = QKVLinear(self.dim, self.qk_dim)
            self.wo = nn.Identity()
        else:
            raise ValueError(f'param_attention mode {self.param_attention} is not surpported!')
        
        self.kv_downsample_mode = kv_downsample_mode
        self.kv_per_win = kv_per_win
        self.kv_downsample_ratio = kv_downsample_ratio
        self.kv_downsample_kenel = kv_downsample_kernel
        if self.kv_downsample_mode == 'ada_avgpool':
            assert self.kv_per_win is not None
            self.kv_down = nn.AdaptiveAvgPool2d(self.kv_per_win)
        elif self.kv_downsample_mode == 'ada_maxpool':
            assert self.kv_per_win is not None
            self.kv_down = nn.AdaptiveMaxPool2d(self.kv_per_win)
        elif self.kv_downsample_mode == 'maxpool':
            assert self.kv_downsample_ratio is not None
            self.kv_down = nn.MaxPool2d(self.kv_downsample_ratio) if self.kv_downsample_ratio > 1 else nn.Identity()
        elif self.kv_downsample_mode == 'avgpool':
            assert self.kv_downsample_ratio is not None
            self.kv_down = nn.AvgPool2d(self.kv_downsample_ratio) if self.kv_downsample_ratio > 1 else nn.Identity()
        elif self.kv_downsample_mode == 'identity': # no kv downsampling
            self.kv_down = nn.Identity()
        elif self.kv_downsample_mode == 'fracpool':
            # assert self.kv_downsample_ratio is not None
            # assert self.kv_downsample_kenel is not None
            # TODO: fracpool
            # 1. kernel size should be input size dependent
            # 2. there is a random factor, need to avoid independent sampling for k and v 
            raise NotImplementedError('fracpool policy is not implemented yet!')
        elif kv_downsample_mode == 'conv':
            # TODO: need to consider the case where k != v so that need two downsample modules
            raise NotImplementedError('conv policy is not implemented yet!')
        else:
            raise ValueError(f'kv_down_sample_mode {self.kv_downsaple_mode} is not surpported!')

        # softmax for local attention
        self.attn_act = nn.Softmax(dim=-1)

        self.auto_pad=auto_pad

    def forward(self, x, ret_attn_mask=False):
        """
        x: NHWC tensor

        Return:
            NHWC tensor
        """
        x = rearrange(x, "n c h w -> n h w c")
         # NOTE: use padding for semantic segmentation
        ###################################################
        if self.auto_pad:
            N, H_in, W_in, C = x.size()

            pad_l = pad_t = 0
            pad_r = (self.n_win - W_in % self.n_win) % self.n_win
            pad_b = (self.n_win - H_in % self.n_win) % self.n_win
            x = F.pad(x, (0, 0, # dim=-1
                          pad_l, pad_r, # dim=-2
                          pad_t, pad_b)) # dim=-3
            _, H, W, _ = x.size() # padded size
        else:
            N, H, W, C = x.size()
            assert H%self.n_win == 0 and W%self.n_win == 0 #
        ###################################################


        # patchify, (n, p^2, w, w, c), keep 2d window as we need 2d pooling to reduce kv size
        x = rearrange(x, "n (j h) (i w) c -> n (j i) h w c", j=self.n_win, i=self.n_win)

        #################qkv projection###################
        # q: (n, p^2, w, w, c_qk)
        # kv: (n, p^2, w, w, c_qk+c_v)
        # NOTE: separte kv if there were memory leak issue caused by gather
        q, kv = self.qkv(x) 

        # pixel-wise qkv
        # q_pix: (n, p^2, w^2, c_qk)
        # kv_pix: (n, p^2, h_kv*w_kv, c_qk+c_v)
        q_pix = rearrange(q, 'n p2 h w c -> n p2 (h w) c')
        kv_pix = self.kv_down(rearrange(kv, 'n p2 h w c -> (n p2) c h w'))
        kv_pix = rearrange(kv_pix, '(n j i) c h w -> n (j i) (h w) c', j=self.n_win, i=self.n_win)

        q_win, k_win = q.mean([2, 3]), kv[..., 0:self.qk_dim].mean([2, 3]) # window-wise qk, (n, p^2, c_qk), (n, p^2, c_qk)

        ##################side_dwconv(lepe)##################
        # NOTE: call contiguous to avoid gradient warning when using ddp
        lepe = self.lepe(rearrange(kv[..., self.qk_dim:], 'n (j i) h w c -> n c (j h) (i w)', j=self.n_win, i=self.n_win).contiguous())
        lepe = rearrange(lepe, 'n c (j h) (i w) -> n (j h) (i w) c', j=self.n_win, i=self.n_win)

        ############ gather q dependent k/v #################

        r_weight, r_idx = self.router(q_win, k_win) # both are (n, p^2, topk) tensors

        kv_pix_sel = self.kv_gather(r_idx=r_idx, r_weight=r_weight, kv=kv_pix) #(n, p^2, topk, h_kv*w_kv, c_qk+c_v)
        k_pix_sel, v_pix_sel = kv_pix_sel.split([self.qk_dim, self.dim], dim=-1)
        # kv_pix_sel: (n, p^2, topk, h_kv*w_kv, c_qk)
        # v_pix_sel: (n, p^2, topk, h_kv*w_kv, c_v)
        
        ######### do attention as normal ####################
        k_pix_sel = rearrange(k_pix_sel, 'n p2 k w2 (m c) -> (n p2) m c (k w2)', m=self.num_heads) # flatten to BMLC, (n*p^2, m, topk*h_kv*w_kv, c_kq//m) transpose here?
        v_pix_sel = rearrange(v_pix_sel, 'n p2 k w2 (m c) -> (n p2) m (k w2) c', m=self.num_heads) # flatten to BMLC, (n*p^2, m, topk*h_kv*w_kv, c_v//m)
        q_pix = rearrange(q_pix, 'n p2 w2 (m c) -> (n p2) m w2 c', m=self.num_heads) # to BMLC tensor (n*p^2, m, w^2, c_qk//m)

        # param-free multihead attention
        attn_weight = (q_pix * self.scale) @ k_pix_sel # (n*p^2, m, w^2, c) @ (n*p^2, m, c, topk*h_kv*w_kv) -> (n*p^2, m, w^2, topk*h_kv*w_kv)
        attn_weight = self.attn_act(attn_weight)
        out = attn_weight @ v_pix_sel # (n*p^2, m, w^2, topk*h_kv*w_kv) @ (n*p^2, m, topk*h_kv*w_kv, c) -> (n*p^2, m, w^2, c)
        out = rearrange(out, '(n j i) m (h w) c -> n (j h) (i w) (m c)', j=self.n_win, i=self.n_win,
                        h=H//self.n_win, w=W//self.n_win)

        out = out + lepe
        # output linear
        out = self.wo(out)

        # NOTE: use padding for semantic segmentation
        # crop padded region
        if self.auto_pad and (pad_r > 0 or pad_b > 0):
            out = out[:, :H_in, :W_in, :].contiguous()

        if ret_attn_mask:
            return out, r_weight, r_idx, attn_weight
        else:
            return rearrange(out, "n h w c -> n c h w")

## ImageConvNet for face and body features extraction

In [7]:
class GLIBlock(nn.Module):
    def __init__(self, channels, ratio,gamma = 2, b = 1):
        super(GLIBlock, self).__init__()
        self.avg_pooling = nn.AdaptiveAvgPool2d(1)
        self.max_pooling = nn.AdaptiveMaxPool2d(1)
        self.fc_layers = nn.Sequential(
            nn.Linear(in_features = channels, out_features = channels // ratio, bias = False),
            nn.ReLU(),
            nn.Linear(in_features = channels // ratio, out_features = channels, bias = False)
        )
        kernel_size = int(abs((math.log(channels, 2) + b) / gamma))
        kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1
        self.conv = nn.Conv1d(1, 1, kernel_size = kernel_size, padding = (kernel_size - 1) // 2, bias = False)

        self.cfc = nn.Parameter(torch.Tensor(channels, 2))
        self.cfc.data.fill_(0)

        self.bn = nn.BatchNorm2d(channels)
        self.activation = nn.Sigmoid()

        setattr(self.cfc, 'srm_param', True)
        setattr(self.bn.weight, 'srm_param', True)
        setattr(self.bn.bias, 'srm_param', True)

        self.sigmoid = nn.Sigmoid()

    def _style_integration(self, t):
        z = t * self.cfc[None, :, :]  # B x C x 2 对应元素相乘
        z = torch.sum(z, dim=2)[:, :, None, None] # B x C x 1 x 1

        # z_hat = self.bn(z)
        g = self.sigmoid(z)

        return g

    def forward(self, x, eps=1e-5):
        b, c, h, w = x.shape # x.shape = torch.Size([8, 512, 14, 14])

        # 全局最大池化
        avg_x_fc = self.avg_pooling(x).view(b, c) # [8, 512]

        # 全局最大池化 + 全局平均池化 + 全局标准差池化(conv1d)
        avg_x_conv = self.avg_pooling(x) # [8, 512,1,1]

        # 特征提取层
        v_fc = self.fc_layers(avg_x_fc)# + self.fc_layers(max_x_fc)  + self.fc_layers(std_x_fc) # [8,512]
        v_conv = self.conv(avg_x_conv.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)# + self.conv(max_x_conv.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1) + self.conv(std_x_conv.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        # [8,512,1,1]

        v_x,v_y = v_fc.shape # 8,512
        v_fc = v_fc.view(v_x,v_y,1) # [8, 512, 1]
        v_conv = v_conv.view(v_x,v_y,1) # [8, 512, 1]

        # 通道平均池化
        v_sum = torch.cat((v_fc, v_conv), dim=2)
        v_sum = self._style_integration(v_sum)
        # print(v_sum.size()) # [2, 3, 2]
        # v_sum = (v_fc + v_conv)/2

        # v = self.sigmoid(v_sum) # [8, 512, 1, 1]
         # [8, 512, 14, 14]
        return x * v_sum

class ImageConvNet_body(nn.Module):
    def __init__(self):
        super(ImageConvNet_body, self).__init__()
        self.pool = nn.MaxPool2d(2, stride=2)

        self.cnn1 = nn.Conv2d(192, 64, 3, stride=2, padding=1)
        self.cnn2 = nn.Conv2d(64, 64, 3, padding=1)
        self.bat10 = nn.BatchNorm2d(64)
        self.bat11 = nn.BatchNorm2d(64)

        self.cnn3 = nn.Conv2d(64, 128, 3, stride=1, padding=1)
        self.cnn4 = nn.Conv2d(128, 128, 3, padding=1)
        self.bat20 = nn.BatchNorm2d(128)
        self.bat21 = nn.BatchNorm2d(128)

        self.cnn5 = nn.Conv2d(128, 256, 3, stride=1, padding=1)
        self.cnn6 = nn.Conv2d(256, 256, 3, padding=1)
        self.bat30 = nn.BatchNorm2d(256)
        self.bat31 = nn.BatchNorm2d(256)

        self.cnn7 = nn.Conv2d(256, 512, 3, stride=1, padding=1)
        self.cnn8 = nn.Conv2d(512, 512, 3, padding=1)
        self.bat40 = nn.BatchNorm2d(512)
        self.bat41 = nn.BatchNorm2d(512)

        # attention
        self.SeBlock1 = GLIBlock(64, 16, gamma=2, b=1)
        self.SeBlock2 = GLIBlock(64, 16, gamma=2, b=1)
        self.SeBlock3 = GLIBlock(128, 16, gamma=2, b=1)
        self.SeBlock4 = GLIBlock(128, 16, gamma=2, b=1)
        self.SeBlock5 = GLIBlock(256, 16, gamma=2, b=1)
        self.SeBlock6 = GLIBlock(256, 16, gamma=2, b=1)
        self.SeBlock7 = GLIBlock(512, 16, gamma=2, b=1)
        self.SeBlock8 = GLIBlock(512, 16, gamma=2, b=1)



    def forward(self, inp):
        c = F.relu(self.SeBlock1(self.bat10(self.cnn1(inp))))
        c = F.relu(self.SeBlock2(self.bat11(self.cnn2(c))))
        # c = self.pool(c)

        c = F.relu(self.SeBlock3(self.bat20(self.cnn3(c))))
        c = F.relu(self.SeBlock4(self.bat21(self.cnn4(c))))
        # c = self.pool(c)

        c = F.relu(self.SeBlock5(self.bat30(self.cnn5(c))))
        c = F.relu(self.SeBlock6(self.bat31(self.cnn6(c))))
        c = self.pool(c)

        c = F.relu(self.SeBlock7(self.bat40(self.cnn7(c))))
        c = F.relu(self.SeBlock8(self.bat41(self.cnn8(c))))

        return c


class ImageConvNet_face(nn.Module):
    def __init__(self):
        super(ImageConvNet_face, self).__init__()
        self.pool = nn.MaxPool2d(2, stride=2)

        self.cnn1 = nn.Conv2d(48, 64, 3, stride=2, padding=1)
        self.cnn2 = nn.Conv2d(64, 64, 3, padding=1)
        self.bat10 = nn.BatchNorm2d(64)
        self.bat11 = nn.BatchNorm2d(64)

        self.cnn3 = nn.Conv2d(64, 128, 3, stride=1, padding=1)
        self.cnn4 = nn.Conv2d(128, 128, 3, padding=1)
        self.bat20 = nn.BatchNorm2d(128)
        self.bat21 = nn.BatchNorm2d(128)

        self.cnn5 = nn.Conv2d(128, 256, 3, stride=1, padding=1)
        self.cnn6 = nn.Conv2d(256, 256, 3, padding=1)
        self.bat30 = nn.BatchNorm2d(256)
        self.bat31 = nn.BatchNorm2d(256)

        self.cnn7 = nn.Conv2d(256, 512, 3, stride=1, padding=1)
        self.cnn8 = nn.Conv2d(512, 512, 3, padding=1)
        self.bat40 = nn.BatchNorm2d(512)
        self.bat41 = nn.BatchNorm2d(512)

        # attention
        self.SeBlock1 = GLIBlock(64, 16, gamma=2, b=1)
        self.SeBlock2 = GLIBlock(64, 16, gamma=2, b=1)
        self.SeBlock3 = GLIBlock(128, 16, gamma=2, b=1)
        self.SeBlock4 = GLIBlock(128, 16, gamma=2, b=1)
        self.SeBlock5 = GLIBlock(256, 16, gamma=2, b=1)
        self.SeBlock6 = GLIBlock(256, 16, gamma=2, b=1)
        self.SeBlock7 = GLIBlock(512, 16, gamma=2, b=1)
        self.SeBlock8 = GLIBlock(512, 16, gamma=2, b=1)

        # self.avg_pool = nn.AdaptiveAvgPool2d(1)
        #
        # self.fc1 = nn.Linear(512, 5)
        # self.fc2 = nn.Linear(512, 7)
        # self.fc3 = nn.Linear(512, 3)
        # self.fc4 = nn.Linear(512, 5)

    def forward(self, inp):
        c = F.relu(self.SeBlock1(self.bat10(self.cnn1(inp))))
        c = F.relu(self.SeBlock2(self.bat11(self.cnn2(c))))
        # c = self.pool(c)

        c = F.relu(self.SeBlock3(self.bat20(self.cnn3(c))))
        c = F.relu(self.SeBlock4(self.bat21(self.cnn4(c))))
        c = self.pool(c)

        c = F.relu(self.SeBlock5(self.bat30(self.cnn5(c))))
        c = F.relu(self.SeBlock6(self.bat31(self.cnn6(c))))
        c = self.pool(c)

        c = F.relu(self.SeBlock7(self.bat40(self.cnn7(c))))
        c = F.relu(self.SeBlock8(self.bat41(self.cnn8(c))))

        return c

## 3D-CNN

In [8]:
class ConvNet3D(nn.Module):
    def __init__(self, num_classes=512, num_keypoints=42):
        super(ConvNet3D, self).__init__()
        self.conv1 = nn.Conv3d(in_channels=3, out_channels=64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        # self.pool = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 1, 1))
        self.fc = nn.Linear(64 * 16 * (num_keypoints) * 1, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        # x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## DBME: Dual-Branch Multimodal Attention

In [9]:
from torch.nn import init

class ECAAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.conv3=nn.Conv1d(1,1,kernel_size=3,padding=(3-1)//2)

        self.attention = nn.MultiheadAttention(embed_dim, num_heads)
        self.sigmoid = nn.Sigmoid()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                init.normal_(m.weight, std=0.001)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

    def forward(self, x):
        y = self.gap(x) # bs, c, 1, 1
        y = y.squeeze(-1).permute(0, 2, 1) # bs, 1, c
        y3=self.conv3(y) #bs,1,c

        attn_output, _ = self.attention(y3, y3, y3) # attn_output: bs, 1,  c
        attn_output = attn_output.permute(0, 2, 1).unsqueeze(-1) # bs, c, 1, 1
        attn_output = self.sigmoid(attn_output) # bs, c, 1, 1
        return x * attn_output.expand_as(x)

class DBME(nn.Module):
    def __init__(self, channels=512,r=4):
        super(DBME, self).__init__()
        inter_channels = int(channels // r) 
        # kernel_size = int(abs((math.log(channels, 2) + 1) / 2))
        # print(kernel_size)
        # print((kernel_size - 1) // 2)
        self.conv1 = nn.Conv2d(channels, inter_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(inter_channels)
        self.local_att1 = nn.Sequential(
            nn.Conv2d(channels, inter_channels, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(inter_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_channels, channels, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(channels),
        )

        # self.conv = nn.Conv1d(1, 1, kernel_size = 3, padding = 1, bias = False)
        self.conv = nn.Conv2d(channels, channels, kernel_size=(3, 3), stride=1, padding=1, bias=False)

        self.bn = nn.BatchNorm2d(channels)

        # self.local_att2 = nn.Sequential(
        #     nn.Conv1d(1, 1, kernel_size = 3, padding =1, bias = False),
        #     nn.BatchNorm2d(channels),
        # )
        self.local_att2 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=(3, 3), stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )


        # self.attention1 = ECABlock(channels, gamma = 2, b = 1)
 
        # self.attention2 = SEBlock(channels, 16)
        self.attention2 = ECAAttention(channels,8)
 
        self.sigmoid = nn.Sigmoid()
 
 
    def forward(self, x, residual):
        xa = x + residual

        xz1 = self.local_att1(xa)
        # xg1 = self.attention1(xz1)
        

        # xz2 = self.conv(xa.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        xz2 = self.conv(xa)
        
        xz2 = self.bn(xz2)
        # xz2 = self.bn(xz2)
        # xg2 = self.attention2(xz2)

        xlg1 = xz2 + xz1
        xlg1 = self.attention2(xlg1)
        wei = self.sigmoid(xlg1)
 
 
        xo = 2 * x * wei + 2 * residual * (1 - wei)
        return xo

## Complete Net

In [10]:
class TotalNet(nn.Module):
    def __init__(self):
        super(TotalNet, self).__init__()
        self.subnet1 = HVAttention(pretrained=False)
        self.subnet2 = HVAttention(pretrained=False)
        self.subnet3 = HVAttention(pretrained=False)
        self.subnet4 = HVAttention(pretrained=False)
        self.subnet5 = HVAttention(pretrained=False)
        self.subnet6 = HVAttention(pretrained=False)

        # Adding BiLevelRoutingAttention after the early subnets
        self.regionAttention1 = BiLevelRoutingAttention(512)
        self.regionAttention2 = BiLevelRoutingAttention(512)
        self.regionAttention3 = BiLevelRoutingAttention(512)
        self.regionAttention4 = BiLevelRoutingAttention(512)
        self.regionAttention5 = BiLevelRoutingAttention(512)
        self.regionAttention6 = BiLevelRoutingAttention(512)

        #self.subnet5 = ImageConvNet_face()
        #self.subnet6 = ImageConvNet_body()

        self.conv3d_gesture = ConvNet3D(num_keypoints=42)
        self.conv3d_posture = ConvNet3D(num_keypoints=26)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # DBME modules
        self.aff_module_for_people_scenes = DBME(channels=512)
        self.aff_module_for_fused_points = DBME(channels=512)

        # ECAAttention layers for channel attention
        self.channel_att1 = ECAAttention(512, 8)
        self.channel_att2 = ECAAttention(512, 8)
        self.channel_att3 = ECAAttention(512, 8)
        self.channel_att4 = ECAAttention(512, 8)

        # Fully connected layers for additional outputs
        self.fc1 = nn.Linear(512, 5)
        self.fc2 = nn.Linear(512, 7)
        self.fc3 = nn.Linear(512, 3)
        self.fc4 = nn.Linear(512, 5)

        self.fc11 = nn.Linear(512, 5)
        self.fc22 = nn.Linear(512, 7)
        self.fc33 = nn.Linear(512, 3)
        self.fc44 = nn.Linear(512, 5)

        # Weights for combining outputs
        self.weight1 = nn.Parameter(torch.tensor(0.5))
        self.weight2 = nn.Parameter(torch.tensor(0.5))
        self.weight3 = nn.Parameter(torch.tensor(0.5))
        self.weight4 = nn.Parameter(torch.tensor(0.5))

    def forward(self, img1, img2, img3, img4, face, body, gesture, posture):

        # Original feature extraction
        h1 = self.regionAttention1(self.subnet1(img1))
        h2 = self.regionAttention2(self.subnet2(img2))
        h3 = self.regionAttention3(self.subnet3(img3))
        h4 = self.regionAttention4(self.subnet4(img4))
        h_face = self.regionAttention5(self.subnet5(face))
        h_body = self.regionAttention6(self.subnet6(body))

        #h_face = F.interpolate(self.subnet5(face), size=(h1.size(2), h1.size(3)), mode='bilinear', align_corners=True)
        #h_body = F.interpolate(self.subnet6(body), size=(h1.size(2), h1.size(3)), mode='bilinear', align_corners=True)

        h_gesture = self.conv3d_gesture(gesture).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 14, 14)
        h_posture = self.conv3d_posture(posture).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 14, 14)

        # Combine all features
        x_people = h1 + h_face + h_body
        x_scenes = h2 + h3 + h4
        x_points = h_gesture + h_posture

        x_all = x_people + x_scenes + x_points

        # Apply ECAAttention to the combined features
        x1 = self.channel_att1(x_all)
        x2 = self.channel_att2(x_all)
        x3 = self.channel_att3(x_all)
        x4 = self.channel_att4(x_all)

        # Fuse with the original fused final feature
        x_fused_people_scenes = self.aff_module_for_people_scenes(x_people, x_scenes)
        x_fused_final = self.aff_module_for_fused_points(x_fused_people_scenes, x_points)
        x_fused_final = self.avg_pool(x_fused_final).view(x_fused_final.size(0), -1)

        # Apply average pooling to ECA-attended features
        x_1 = self.avg_pool(x1).view(x1.size(0), -1)
        x_2 = self.avg_pool(x2).view(x2.size(0), -1)
        x_3 = self.avg_pool(x3).view(x3.size(0), -1)
        x_4 = self.avg_pool(x4).view(x4.size(0), -1)

        # Apply fully connected layers to get initial outputs
        out01 = self.fc1(x_fused_final)
        out02 = self.fc2(x_fused_final)
        out03 = self.fc3(x_fused_final)
        out04 = self.fc4(x_fused_final)

        out11 = self.fc11(x_1)
        out22 = self.fc22(x_2)
        out33 = self.fc33(x_3)
        out44 = self.fc44(x_4)

        # Weighted combination of initial and ECA-attended outputs
        out1 = torch.sigmoid(self.weight1) * out01 + (1 - torch.sigmoid(self.weight1)) * out11
        out2 = torch.sigmoid(self.weight2) * out02 + (1 - torch.sigmoid(self.weight2)) * out22
        out3 = torch.sigmoid(self.weight3) * out03 + (1 - torch.sigmoid(self.weight3)) * out33
        out4 = torch.sigmoid(self.weight4) * out04 + (1 - torch.sigmoid(self.weight4)) * out44

        return out1, out2, out3, out4

# Training procedure

In [11]:
from prettytable import PrettyTable
from matplotlib import pyplot as plt

class valConfusionMatrix(object):
    def __init__(self, num_classes: int, labels: list):
        self.matrix = np.zeros((num_classes, num_classes))
        self.num_classes = num_classes
        self.labels = labels

    def update(self, preds, labels):
        for p, t in zip(preds, labels):
            self.matrix[p, t] += 1

    def summary(self):
        f1_list = []
        for i in range(self.num_classes):
            TP = self.matrix[i, i]
            FP = np.sum(self.matrix[i, :]) - TP
            FN = np.sum(self.matrix[:, i]) - TP
            TN = np.sum(self.matrix) - TP - FP - FN
            Precision = round(TP / (TP + FP), 3) if TP + FP != 0 else 0.
            Recall = round(TP / (TP + FN), 3) if TP + FN != 0 else 0.
            # Specificity = round(TN / (TN + FP), 3) if TN + FP != 0 else 0.
            F1 = round(2 * Precision * Recall / (Precision + Recall), 3) if Precision + Recall != 0 else 0.
            f1_list.append(F1)
        return f1_list


class testConfusionMatrix(object):
    def __init__(self, num_classes: int, labels: list):
        self.matrix = np.zeros((num_classes, num_classes))
        self.num_classes = num_classes
        self.labels = labels

    def update(self, preds, labels):
        for p, t in zip(preds, labels):
            self.matrix[p, t] += 1

    def summary(self):
        # calculate accuracy
        sum_TP = 0
        for i in range(self.num_classes):
            sum_TP += self.matrix[i, i]
        acc = sum_TP / np.sum(self.matrix)
        print("the model accuracy is ", acc)

        # precision, recall, specificity
        table = PrettyTable()
        table.field_names = ["", "Precision", "Recall", "Specificity", "F1"]
        for i in range(self.num_classes):
            TP = self.matrix[i, i]
            FP = np.sum(self.matrix[i, :]) - TP
            FN = np.sum(self.matrix[:, i]) - TP
            TN = np.sum(self.matrix) - TP - FP - FN
            Precision = round(TP / (TP + FP), 3) if TP + FP != 0 else 0.
            Recall = round(TP / (TP + FN), 3) if TP + FN != 0 else 0.
            Specificity = round(TN / (TN + FP), 3) if TN + FP != 0 else 0.
            F1 = round(2 * Precision * Recall / (Precision + Recall), 3) if Precision + Recall != 0 else 0.
            table.add_row([self.labels[i], Precision, Recall, Specificity, F1])
        print(table)

    def plot(self):
        matrix = self.matrix
        print(matrix)
        plt.imshow(matrix, cmap=plt.cm.Blues)

        # 设置x轴坐标label
        plt.xticks(range(self.num_classes), self.labels, rotation=45)
        # 设置y轴坐标label
        plt.yticks(range(self.num_classes), self.labels)
        # 显示colorbar
        plt.colorbar()
        plt.xlabel('True Labels')
        plt.ylabel('Predicted Labels')
        plt.title('Confusion matrix')

        # 在图中标注数量/概率信息
        thresh = matrix.max() / 2
        for x in range(self.num_classes):
            for y in range(self.num_classes):
                # 注意这里的matrix[y, x]不是matrix[x, y]
                info = int(matrix[y, x])
                plt.text(x, y, info,
                         verticalalignment='center',
                         horizontalalignment='center',
                         color="white" if info > thresh else "black")
        plt.tight_layout()
        plt.show()


class LossAverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


class AccAverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val
        self.count += n

    def getacc(self):
        return (self.sum * 100) / self.count

In [ ]:
from torch.optim import SGD
import torchvision
import time

EPOCHS = 50
use_cuda = torch.cuda.is_available()

model = TotalNet()
model = nn.DataParallel(model)

model = model.cuda()

crossEntropy1 = nn.CrossEntropyLoss()
crossEntropy2 = nn.CrossEntropyLoss()
crossEntropy3 = nn.CrossEntropyLoss()
crossEntropy4 = nn.CrossEntropyLoss()
print("Loaded dataloader and loss function.")

best_precision = 0
lowest_loss = 100000
best_avgf1 = 0
best_weightf1 = 0

def load_checkpoint(model, optim, checkpoint_path):
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['state_dict'])
        optim.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed training from epoch {start_epoch}")
        return start_epoch
    else:
        print("No checkpoint found, starting from scratch")
        return 0

def lr_for_epoch(epoch):
    if epoch <= 8:
        return 1e-3
    elif epoch <= 16:
        return 0.5e-3
    elif epoch <= 25:
        return 0.5e-4
    elif epoch <= 33:
        return 1e-5
    else:
        return 0.5e-5

"""def lr_for_epoch(epoch):
    if epoch <= 20:
        return 1e-3
    elif epoch <= 35:
        return 1e-5
    else:
        return 0.1e-5"""
    

checkpoint_path = r'checkpoint\best_model_CNNTrans_basic_v5.pt'

optim = SGD(model.parameters(), lr=lr_for_epoch(0), momentum=0.9, weight_decay=1e-4)
start_epoch = load_checkpoint(model, optim, checkpoint_path)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    lr = lr_for_epoch(epoch)
    for group in optim.param_groups:
        group['lr'] = lr

    train_losses = LossAverageMeter()

    # train_acc1 是一个AccAverageMeter类的实例，用于追踪训练准确率的平均值
    train_acc1 = AccAverageMeter()
    train_acc2 = AccAverageMeter()
    train_acc3 = AccAverageMeter()
    train_acc4 = AccAverageMeter()
    if (epoch == 0):
        end = time.time()
    # for subepoch, (img, aud, out) in enumerate(train_dataloader):#dataloader context,behavior_label

    # 遍历训练数据加载器，获取图像img1, img2, img3, img4和相应标签 , points
    for subepoch, (img1,img2,img3,img4,face,body, posture, gesture, emotion_label, behavior_label, context_label, vehicle_label) in enumerate(
            train_dataloader):

        # 打印第一个epoch中第一个批次所用时间
        if (epoch == 0 and subepoch == 0):
            print(time.time() - end)
        # 梯度清零
        optim.zero_grad()
        # 改变图像张量形状（五维->四维），使其与模型兼容
        B, _, _, H, W = img1.shape
        img1 = img1.view(B, -1,  H, W)  # [16, 3, 16, 224, 224]
        img2 = img2.view(B, -1,  H, W)
        img3 = img3.view(B, -1,  H, W)
        img4 = img4.view(B, -1,  H, W)

        # Flatten the temporal/channel dimensions before resizing.
        face = face.view(B, -1, face.shape[-2], face.shape[-1])
        face = torchvision.transforms.Resize((224, 224)).to(device, dtype=torch.float32, non_blocking=True)(face)
        body = body.view(B, -1,  H, W)
        #face = face.view(B, -1,  64, 64)
        #body = body.view(B, -1,  112,112)
# Gesture Skeleton Keypoint 3 (C)×16 (F)×42 (K)×1 (P)
# Posture Skeleton Keypoint 3 (C)×16 (F)×26 (K)×1 (P)
        gesture = gesture.view(B, 3, 16, 42, 1)
        posture = posture.view(B,3, 16, 26, 1)    
        # 批次大小
        M = img1.shape[0]
        # 将数据移到GPU上
        if use_cuda:
            img1 = img1.to(device, dtype=torch.float32, non_blocking=True)
            img2 = img2.to(device, dtype=torch.float32, non_blocking=True)
            img3 = img3.to(device, dtype=torch.float32, non_blocking=True)
            img4 = img4.to(device, dtype=torch.float32, non_blocking=True)
            
            face = face.to(device, dtype=torch.float32, non_blocking=True)
            body = body.to(device, dtype=torch.float32, non_blocking=True)
            
            gesture = gesture.to(device, dtype=torch.float32, non_blocking=True)
            posture = posture.to(device, dtype=torch.float32, non_blocking=True)
            
        emotion_label = emotion_label.to(device, non_blocking=True)
        behavior_label = behavior_label.to(device, non_blocking=True)
        context_label = context_label.to(device, non_blocking=True)
        vehicle_label = vehicle_label.to(device, non_blocking=True)

        # 将输入图像传递到模型(前向传播)
        out1, out2, out3, out4 = model(img1,img2,img3,img4,face,body, gesture, posture)
        # if subepoch%400 == 0:
        # 	print(o)
        # 	print(out)
        # print(o.shape, out.shape)
        # print(out1.shape, emotion_label.shape)

        # 计算单独损失和总损失
        loss1 = crossEntropy1(out1, emotion_label)
        # print(out2.shape, behavior_label.shape)
        loss2 = crossEntropy2(out2, behavior_label)
        # print(out3.shape, context_label.shape)
        loss3 = crossEntropy3(out3, context_label)
        # print(out4.shape, vehicle_label.shape)
        loss4 = crossEntropy4(out4, vehicle_label)
        loss = loss1 + loss2 + loss3 + loss4
        # print(loss)

        # 更新训练损失
        train_losses.update(loss.item(), M)
        # 反向传播，进一步优化
        loss.backward()
        optim.step()

        # Calculate accuracy
        out1 = F.softmax(out1, 1)  # Softmax将一组数值转换为概率分布
        ind = out1.argmax(dim=1)  # ind保存每行中最大值所在的索引，即概率最大的类别
        # print(ind.data)
        # print(out1.data)

        # 计算当前批次的准确率
        accuracy1 = (ind.data == emotion_label.data).sum() * 1.0 / M
        # 更新整个训练过程的准确率平均值
        train_acc1.update((ind.data == emotion_label.data).sum() * 1.0, M)

        out2 = F.softmax(out2, 1)
        ind = out2.argmax(dim=1)
        accuracy2 = (ind.data == behavior_label.data).sum() * 1.0 / M
        train_acc2.update((ind.data == behavior_label.data).sum() * 1.0, M)

        out3 = F.softmax(out3, 1)
        ind = out3.argmax(dim=1)
        accuracy3 = (ind.data == context_label.data).sum() * 1.0 / M
        train_acc3.update((ind.data == context_label.data).sum() * 1.0, M)

        out4 = F.softmax(out4, 1)
        ind = out4.argmax(dim=1)
        accuracy4 = (ind.data == vehicle_label.data).sum() * 1.0 / M
        train_acc4.update((ind.data == vehicle_label.data).sum() * 1.0, M)


        # if subepoch % 1 == 0:
        print("Epoch: %d, Subepoch: %d, Loss: %f, "
                "batch_size: %d, total_acc1: %f, total_acc2: %f, total_acc3: %f, total_acc4: %f" % (
            epoch, subepoch, train_losses.avg, M,
            train_acc1.getacc(),
            train_acc2.getacc(),
            train_acc3.getacc(),
            train_acc4.getacc()))
        ################################################################################################################################## 
        #with open(file="/root/AIDE/CNNTrans_basic_v5.txt", mode="a+") as f:
        #    f.write("Epoch: %d, Subepoch: %d, Loss: %f, batch_size: %d, total_acc1: %f,total_acc2: %f, total_acc3: %f, total_acc4: %f"\
        #            %(epoch, subepoch, train_losses.avg, M, train_acc1.getacc(), train_acc2.getacc(), train_acc3.getacc(), train_acc4.getacc()))
        ####################################################################################################################################
        

    # 验证阶段
    print("Valing...")
    # val_losses = LossAverageMeter()

    val_losses1 = LossAverageMeter()
    val_losses2 = LossAverageMeter()
    val_losses3 = LossAverageMeter()
    val_losses4 = LossAverageMeter()
    # val_losses = (val_losses1.avg + val_losses2.avg + val_losses3.avg + val_losses4.avg) / 4.0

    val_acc1 = AccAverageMeter()
    val_acc2 = AccAverageMeter()
    val_acc3 = AccAverageMeter()
    val_acc4 = AccAverageMeter()

    # 混淆矩阵
    valconfusion1 = valConfusionMatrix(num_classes = 5, labels = EMOTION_LABEL)
    valconfusion2 = valConfusionMatrix(num_classes = 7, labels = DRIVER_BEHAVIOR_LABEL)
    valconfusion3 = valConfusionMatrix(num_classes = 3, labels = SCENE_CENTRIC_CONTEXT_LABEL)
    valconfusion4 = valConfusionMatrix(num_classes = 5, labels = VEHICLE_BASED_CONTEXT_LABEL)

    # 将模型设置为评估模式
    model.eval()

    for subepoch1, (img1,img2,img3,img4,face,body, posture, gesture,emotion_label, behavior_label, context_label, vehicle_label) in enumerate(
            val_dataloader):

        if (epoch == 0 and subepoch1 == 0):
            print(time.time() - end)
        with torch.no_grad():
                            
            B, _, _, H, W = img1.shape
            img1 = img1.view(B, -1,  H, W)  # [16, 3, 16, 224, 224]
            img2 = img2.view(B, -1,  H, W)
            img3 = img3.view(B, -1,  H, W)
            img4 = img4.view(B, -1,  H, W)

            # Flatten the temporal/channel dimensions before resizing.
            face = face.view(B, -1, face.shape[-2], face.shape[-1])
            face = torchvision.transforms.Resize((224, 224)).to(device, dtype=torch.float32, non_blocking=True)(face)
            body = body.view(B, -1,  H, W)
            #face = face.view(B, -1,  64, 64)
            #body = body.view(B, -1,  112,112)
# Gesture Skeleton Keypoint 3 (C)×16 (F)×42 (K)×1 (P)
# Posture Skeleton Keypoint 3 (C)×16 (F)×26 (K)×1 (P)
            gesture = gesture.view(B, 3,16,  42, 1)
            posture = posture.view(B,3,16, 26, 1)    
            # gesture = gesture.view(B, -1,  26, 1)
            # posture = posture.view(B,-1, 21, 1)                 

            # 批次大小
            M = img1.shape[0]
            # 将数据移到GPU上
            if use_cuda:
                img1 = img1.to(device, dtype=torch.float32, non_blocking=True)
                img2 = img2.to(device, dtype=torch.float32, non_blocking=True)
                img3 = img3.to(device, dtype=torch.float32, non_blocking=True)
                img4 = img4.to(device, dtype=torch.float32, non_blocking=True)

                face = face.to(device, dtype=torch.float32, non_blocking=True)
                body = body.to(device, dtype=torch.float32, non_blocking=True)

                gesture = gesture.to(device, dtype=torch.float32, non_blocking=True)
                posture = posture.to(device, dtype=torch.float32, non_blocking=True)

            emotion_label = emotion_label.to(device, non_blocking=True)
            behavior_label = behavior_label.to(device, non_blocking=True)
            context_label = context_label.to(device, non_blocking=True)
            vehicle_label = vehicle_label.to(device, non_blocking=True)

            # 将输入图像传递到模型(前向传播)
            out1, out2, out3, out4 = model(img1,img2,img3,img4,face,body,gesture,posture)
            

            loss1 = crossEntropy1(out1, emotion_label)
            # print(out2.shape, behavior_label.shape)
            loss2 = crossEntropy2(out2, behavior_label)
            # print(out3.shape, context_label.shape)
            loss3 = crossEntropy3(out3, context_label)
            # print(out4.shape, vehicle_label.shape)
            loss4 = crossEntropy4(out4, vehicle_label)
            loss = loss1 + loss2 + loss3 + loss4
            # print(loss)
            val_losses1.update(loss1.item(), M)
            val_losses2.update(loss2.item(), M)
            val_losses3.update(loss3.item(), M)
            val_losses4.update(loss4.item(), M)

            val_losses = (val_losses1.avg + val_losses2.avg + val_losses3.avg + val_losses4.avg) / 4.0

            ########################################################## Early stopping mechanism (commented out)
            #早停，防止过拟合
            #early_stopping = EarlyStopping(7, verbose=True)
            #early_stopping(val_losses, model)
            # 若满足 early stopping 要求
            #if early_stopping.early_stop:
            #    print("Early stopping")
            #    # 结束模型训练
            #    break
            ###########################################################

            # Calculate accuracy
            out1 = F.softmax(out1, 1)
            ind1 = out1.argmax(dim=1)
            # print(ind.data)
            # print(out1.data)
            accuracy1 = (ind1.data == emotion_label.data).sum() * 1.0 / M
            val_acc1.update((ind1.data == emotion_label.data).sum() * 1.0, M)
            valconfusion1.update(ind1.to("cpu").numpy(), emotion_label.to("cpu").numpy())  # 更新混淆矩阵
            avgf11 = (valconfusion1.summary()[0] + valconfusion1.summary()[1] + valconfusion1.summary()[2]+
                        valconfusion1.summary()[3]+valconfusion1.summary()[4]) / 5.0

            out2 = F.softmax(out2, 1)
            ind2 = out2.argmax(dim=1)
            accuracy2 = (ind2.data == behavior_label.data).sum() * 1.0 / M
            val_acc2.update((ind2.data == behavior_label.data).sum() * 1.0, M)
            valconfusion2.update(ind2.to("cpu").numpy(), behavior_label.to("cpu").numpy())
            avgf12 = (valconfusion2.summary()[0] + valconfusion2.summary()[1] + valconfusion2.summary()[2] +
                        valconfusion2.summary()[3] + valconfusion2.summary()[4] +
                        valconfusion2.summary()[5] + valconfusion2.summary()[6]) / 7.0

            out3 = F.softmax(out3, 1)
            ind3 = out3.argmax(dim=1)
            accuracy3 = (ind3.data == context_label.data).sum() * 1.0 / M
            val_acc3.update((ind3.data == context_label.data).sum() * 1.0, M)
            valconfusion3.update(ind3.to("cpu").numpy(), context_label.to("cpu").numpy())
            avgf13 = (valconfusion3.summary()[0] + valconfusion3.summary()[1] + valconfusion3.summary()[2]) / 3.0

            out4 = F.softmax(out4, 1)
            ind4 = out4.argmax(dim=1)
            accuracy4 = (ind4.data == vehicle_label.data).sum() * 1.0 / M
            val_acc4.update((ind4.data == vehicle_label.data).sum() * 1.0, M)
            valconfusion4.update(ind4.to("cpu").numpy(), vehicle_label.to("cpu").numpy())
            avgf14 = (valconfusion4.summary()[0] + valconfusion4.summary()[1] + valconfusion4.summary()[2] +
                        valconfusion4.summary()[3] + valconfusion4.summary()[4]) / 5.0

            total_avgf1 = (avgf11 + avgf12 + avgf13 + avgf14) / 4.0

            # if subepoch1 % 1 == 0:
            print(
                "Val Epoch: %d, Subepoch: %d, Loss: %f, batch_size: %d,total_acc1: %f, total_acc2: %f, "
                "total_acc3: %f, total_acc4: %f, avgf11: %f, avgf12: %f, avgf13: %f, avgf14: %f" % (
                    epoch, subepoch1, val_losses, M,
                    val_acc1.getacc(),
                    val_acc2.getacc(),
                    val_acc3.getacc(),
                    val_acc4.getacc(), avgf11, avgf12, avgf13, avgf14))
            #################################################################################################################################################################
            #with open(file="/root/AIDE/val_CNNTrans_basic_v5.txt", mode="a+") as f:
                #f.write("Epoch: %d, Subepoch: %d, Loss: %f, batch_size: %d, total_acc1: %f,total_acc2: %f, total_acc3: %f, total_acc4: %f, \
                    #    avgf11: %f, avgf12: %f, avgf13: %f, avgf14: %f\n"\
                    #%(epoch, subepoch1, val_losses, M, val_acc1.getacc(), val_acc2.getacc(), val_acc3.getacc(), val_acc4.getacc(),avgf11, avgf12, avgf13, avgf14))
            #################################################################################################################################################################


    # 更新最佳模型
    is_best_avgf1 = total_avgf1 > best_avgf1
    # is_best_weightf1 = weightf1 > best_weightf1
    val_acc = (val_acc1.getacc() + val_acc2.getacc() + val_acc3.getacc() + val_acc4.getacc()) / 4.0
    is_best = val_acc > best_precision
    is_lowest_loss = val_losses < lowest_loss
    best_precision = max(val_acc, best_precision)
    lowest_loss = min(val_losses, lowest_loss)
    best_avgf1 = max(total_avgf1, best_avgf1)


    print("Epoch: %d,best_precision: %f,lowest_loss: %f,best_avgf1: %f" % (
    epoch, best_precision, lowest_loss, best_avgf1))

    checkpoint_dir = './checkpoint'
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    # 保存最佳模型(将当前模型的权重保存到best_model.pt文件中)
    best_path = os.path.join(checkpoint_dir, 'best_model_CNNTrans_basic_v5.pt')
    if is_best:
        #with open(file="/root/val_CNNTrans_basic_v5.txt", mode="w") as f:
        #        f.write("Epoch: %d, Subepoch: %d, Loss: %f, batch_size: %d, total_acc1: %f,total_acc2: %f, total_acc3: %f, total_acc4: %f, \
        #                avgf11: %f, avgf12: %f, avgf13: %f, avgf14: %f\n"\
        #            %(epoch, subepoch1, val_losses, M, val_acc1.getacc(), val_acc2.getacc(), val_acc3.getacc(), val_acc4.getacc(),avgf11, avgf12, avgf13, avgf14))
        # shutil.copyfile(save_path, best_path)
        torch.save({
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
        }, best_path)
        print("Successfully saved the model with the best precision!")

    # 保存最低损失模型
    # lowest_path = os.path.join(checkpoint_dir, 'lowest_loss_swin_block_context.pt')
    # if is_lowest_loss:
    #     shutil.copyfile(save_path, lowest_path)
        # torch.save(model.state_dict(), lowest_path)
    #    print("Successfully saved the model with the lowest loss!")

    # 保存最佳平均F1模型
    # best_avgf1_path = os.path.join(checkpoint_dir, 'best_avgf1_swin_block_context.pt')
    # if is_best_avgf1:
    #     shutil.copyfile(save_path, best_avgf1_path)
        # torch.save(model.state_dict(), best_avgf1_path)
    #    print("Successfully saved the model with the best avgf1!")

Loaded dataloader and loss function.
No checkpoint found, starting from scratch
13.853699207305908
Epoch: 0, Subepoch: 0, Loss: 6.226079, batch_size: 12, total_acc1: 25.000000, total_acc2: 50.000000, total_acc3: 16.666668, total_acc4: 25.000000
Epoch: 0, Subepoch: 1, Loss: 6.349461, batch_size: 12, total_acc1: 29.166668, total_acc2: 33.333336, total_acc3: 37.500000, total_acc4: 41.666668
Epoch: 0, Subepoch: 2, Loss: 5.987166, batch_size: 12, total_acc1: 41.666668, total_acc2: 30.555555, total_acc3: 50.000000, total_acc4: 47.222221
Epoch: 0, Subepoch: 3, Loss: 5.781066, batch_size: 12, total_acc1: 47.916668, total_acc2: 27.083334, total_acc3: 56.250000, total_acc4: 50.000000
Epoch: 0, Subepoch: 4, Loss: 5.827974, batch_size: 12, total_acc1: 51.666668, total_acc2: 31.666668, total_acc3: 58.333336, total_acc4: 43.333336
Epoch: 0, Subepoch: 5, Loss: 5.897878, batch_size: 12, total_acc1: 51.388889, total_acc2: 34.722221, total_acc3: 61.111111, total_acc4: 44.444447
Epoch: 0, Subepoch: 6, Lo

# Test split

In [15]:
class TestMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val
        self.count += n

    def getacc(self):
        return (self.sum * 100) / self.count

def test(use_cuda=True, batch_size=16, model_name="F:/Stage Project/code/BaselineReproduction/checkpoint/best_model_CNNTrans_basic_v5.pt"):

    model = TotalNet()  # 创建模型实例
    model = nn.DataParallel(model)  # 使用 DataParallel 包装模型
    model = model.cuda()  # 将模型移动到 CUDA 上
    if os.path.exists(model_name):
        model.load_state_dict(torch.load(model_name)['state_dict'])
        print("Loading from previous checkpoint.")
        

    test_dataset = CarDataset(csv_file=f'{DATASET_PATH}/testing.csv', dataset_path=DATASET_PATH, horizontal_flip_prob=0.0, vertical_flip_prob=0.0)

    test_dataloader = DataLoader(test_dataset, batch_size=6, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)

    crossEntropy = nn.CrossEntropyLoss()
    print("Loaded dataloader and loss function.")

    test_losses = LossAverageMeter()
    test_acc1 = TestMeter()
    test_acc2 = TestMeter()
    test_acc3 = TestMeter()
    test_acc4 = TestMeter()


    testconfusion1 = valConfusionMatrix(num_classes=5, labels=EMOTION_LABEL)
    testconfusion2 = valConfusionMatrix(num_classes=7, labels=DRIVER_BEHAVIOR_LABEL)
    testconfusion3 = valConfusionMatrix(num_classes=3, labels=SCENE_CENTRIC_CONTEXT_LABEL)
    testconfusion4 = valConfusionMatrix(num_classes=5, labels=VEHICLE_BASED_CONTEXT_LABEL)

    model.eval()

    for subepoch2, (img1,img2,img3,img4,face,body,posture, gesture,emotion_label, behavior_label, context_label, vehicle_label) in enumerate(
                test_dataloader):

        # if (epoch == 0 and subepoch2 == 0):
        #     print(time.time() - end)
        with torch.no_grad():

            B, _, _, H, W = img1.shape
            img1 = img1.view(B, -1,  H, W)  # [16, 3, 16, 224, 224]
            img2 = img2.view(B, -1,  H, W)
            img3 = img3.view(B, -1,  H, W)
            img4 = img4.view(B, -1,  H, W)

            # Flatten the temporal/channel dimensions before resizing.
            face = face.view(B, -1, face.shape[-2], face.shape[-1])
            face = torchvision.transforms.Resize((224, 224)).to(device, dtype=torch.float32, non_blocking=True)(face)
            body = body.view(B, -1,  H, W)
            #face = face.view(B, -1,  64, 64)
            #body = body.view(B, -1,  112,112)
# Gesture Skeleton Keypoint 3 (C)×16 (F)×42 (K)×1 (P)
# Posture Skeleton Keypoint 3 (C)×16 (F)×26 (K)×1 (P)
            gesture = gesture.view(B, 3,16,  42, 1)
            posture = posture.view(B,3,16, 26, 1)    
            # gesture = gesture.view(B, -1,  26, 1)
            # posture = posture.view(B,-1, 21, 1)             

            # 批次大小
            M = img1.shape[0]
            # 将数据移到GPU上
            if use_cuda:
                img1 = img1.to(device, dtype=torch.float32, non_blocking=True)
                img2 = img2.to(device, dtype=torch.float32, non_blocking=True)
                img3 = img3.to(device, dtype=torch.float32, non_blocking=True)
                img4 = img4.to(device, dtype=torch.float32, non_blocking=True)

                face = face.to(device, dtype=torch.float32, non_blocking=True)
                body = body.to(device, dtype=torch.float32, non_blocking=True)

                gesture = gesture.to(device, dtype=torch.float32, non_blocking=True)
                posture = posture.to(device, dtype=torch.float32, non_blocking=True)

            emotion_label = emotion_label.to(device, non_blocking=True)
            behavior_label = behavior_label.to(device, non_blocking=True)
            context_label = context_label.to(device, non_blocking=True)
            vehicle_label = vehicle_label.to(device, non_blocking=True)

            # 将输入图像传递到模型(前向传播)
            out1, out2, out3, out4 = model(img1,img2,img3,img4,face,body,gesture,posture)


            # Calculate accuracy
            out1 = F.softmax(out1, 1)
            ind1 = out1.argmax(dim=1)
            # print(ind.data)
            # print(out1.data)
            accuracy1 = (ind1.data == emotion_label.data).sum() * 1.0 / M
            test_acc1.update((ind1.data == emotion_label.data).sum() * 1.0, M)
            testconfusion1.update(ind1.to("cpu").numpy(), emotion_label.to("cpu").numpy())
            avgf11 = (testconfusion1.summary()[0] + testconfusion1.summary()[1] + testconfusion1.summary()[2] +
                      testconfusion1.summary()[3] + testconfusion1.summary()[4]) / 5.0

            out2 = F.softmax(out2, 1)
            ind2 = out2.argmax(dim=1)
            accuracy2 = (ind2.data == behavior_label.data).sum() * 1.0 / M
            test_acc2.update((ind2.data == behavior_label.data).sum() * 1.0, M)
            testconfusion2.update(ind2.to("cpu").numpy(), behavior_label.to("cpu").numpy())
            avgf12 = (testconfusion2.summary()[0] + testconfusion2.summary()[1] + testconfusion2.summary()[2] +
                      testconfusion2.summary()[3] + testconfusion2.summary()[4] +
                      testconfusion2.summary()[5] + testconfusion2.summary()[6]) / 7.0

            out3 = F.softmax(out3, 1)
            ind3 = out3.argmax(dim=1)
            accuracy3 = (ind3.data == context_label.data).sum() * 1.0 / M
            test_acc3.update((ind3.data == context_label.data).sum() * 1.0, M)
            testconfusion3.update(ind3.to("cpu").numpy(), context_label.to("cpu").numpy())
            avgf13 = (testconfusion3.summary()[0] + testconfusion3.summary()[1] + testconfusion3.summary()[2]) / 3.0

            out4 = F.softmax(out4, 1)
            ind4 = out4.argmax(dim=1)
            accuracy4 = (ind4.data == vehicle_label.data).sum() * 1.0 / M
            test_acc4.update((ind4.data == vehicle_label.data).sum() * 1.0, M)
            testconfusion4.update(ind4.to("cpu").numpy(), vehicle_label.to("cpu").numpy())
            avgf14 = (testconfusion4.summary()[0] + testconfusion4.summary()[1] + testconfusion4.summary()[2] +
                      testconfusion4.summary()[3] + testconfusion4.summary()[4]) / 5.0

            total_avgf1 = (avgf11 + avgf12 + avgf13 + avgf14) / 4.0

            # if subepoch1 % 1 == 0:
            print(
                "Test  Subepoch: %d, batch_size: %d,total_acc1: %f, total_acc2: %f, "
                "total_acc3: %f, total_acc4: %f, avgf11: %f, avgf12: %f, avgf13: %f, avgf14: %f" % (
                    subepoch2, M,
                    test_acc1.getacc(),
                    test_acc2.getacc(),
                    test_acc3.getacc(),
                    test_acc4.getacc(), avgf11, avgf12, avgf13, avgf14))
            
            """with open(file="/root/AIDE/test_CNNTrans_basic_v5.txt", mode="a+") as f:
                    f.write("Test  Subepoch: %d, batch_size: %d,total_acc1: %f, total_acc2: %f, "
                "total_acc3: %f, total_acc4: %f, avgf11: %f, avgf12: %f, avgf13: %f, avgf14: %f\n"\
                     %(subepoch2, M,
                    test_acc1.getacc(),
                    test_acc2.getacc(),
                    test_acc3.getacc(),
                    test_acc4.getacc(), avgf11, avgf12, avgf13, avgf14))"""


    testconfusion1.summary()
    testconfusion2.summary()
    testconfusion3.summary()
    testconfusion4.summary()

test(use_cuda=True, batch_size=16, model_name="F:/Stage Project/code/BaselineReproduction/checkpoint/best_model_CNNTrans_basic_v5.pt")

Loading from previous checkpoint.
Loaded dataloader and loss function.
Test  Subepoch: 0, batch_size: 6,total_acc1: 83.333336, total_acc2: 66.666672, total_acc3: 100.000000, total_acc4: 83.333336, avgf11: 0.181800, avgf12: 0.217714, avgf13: 0.666667, avgf14: 0.493400
Test  Subepoch: 1, batch_size: 6,total_acc1: 83.333336, total_acc2: 58.333336, total_acc3: 100.000000, total_acc4: 75.000000, avgf11: 0.280000, avgf12: 0.265286, avgf13: 0.666667, avgf14: 0.487200
Test  Subepoch: 2, batch_size: 6,total_acc1: 77.777779, total_acc2: 61.111111, total_acc3: 100.000000, total_acc4: 72.222221, avgf11: 0.357800, avgf12: 0.416429, avgf13: 0.666667, avgf14: 0.461600
Test  Subepoch: 3, batch_size: 6,total_acc1: 66.666672, total_acc2: 62.500000, total_acc3: 91.666672, total_acc4: 79.166672, avgf11: 0.306600, avgf12: 0.392857, avgf13: 0.649000, avgf14: 0.508400
Test  Subepoch: 4, batch_size: 6,total_acc1: 66.666672, total_acc2: 60.000004, total_acc3: 90.000008, total_acc4: 80.000008, avgf11: 0.438200,